# LLM Playground on Isambard

Load **NVIDIA Nemotron 3 Super (120B-A12B)** across the four GH200 GPUs of a single
Isambard node and generate text with the HuggingFace `transformers` pipeline.

**Before you run this**, set the environment up once from a terminal on a compute node:

```bash
bash setup_environment.sh
```

Then pick this repo's `.venv` as the notebook kernel:
*Ctrl/Cmd+Shift+P -> "Notebook: Select Kernel" -> `.venv/bin/python`*.

---


## 1. Environment check

Run this first. It takes two seconds and tells you immediately whether the two
classic Isambard traps have bitten you.


In [1]:
import os

# --- Trap 1: an inherited LD_PRELOAD ---------------------------------------
# Isambard job scripts often export LD_PRELOAD=/tools/brics/.../libnccl.so.
# That forces the SYSTEM NCCL ahead of the one bundled inside the torch wheel;
# the system copy is older and lacks symbols torch needs, so `import torch`
# dies with "undefined symbol: nccl...". It is a plain env var, so it SURVIVES
# `module purge`/`module reset` -- it must be unset explicitly.
if os.environ.pop("LD_PRELOAD", None):
    print("Cleared an inherited LD_PRELOAD (this would have broken import torch)")

# Use the SHARED model cache rather than downloading another copy.
os.environ.setdefault("HF_HOME", "/projects/a5k/public/hf")

import torch, transformers

print(f"torch          {torch.__version__}")
print(f"transformers   {transformers.__version__}")
print(f"CUDA available {torch.cuda.is_available()}")
print(f"GPUs visible   {torch.cuda.device_count()}")

# --- Trap 2: a CPU-only or CUDA-13 wheel -----------------------------------
# Isambard is ARM (aarch64). Plain `pip install torch` gives you either a
# CPU-ONLY aarch64 build (silently no GPU, no error) or, more recently, a
# CUDA-13 build that needs a newer driver than this cluster has. Both look
# fine until they don't. pyproject.toml pins the cu126 index to avoid this.
assert "+cu" in torch.__version__, "CPU-only wheel! See pyproject.toml [tool.uv.sources]."
assert torch.cuda.is_available(), "No GPU. Are you on a compute node with --gpus-per-node=4?"

total = sum(torch.cuda.get_device_properties(i).total_memory
            for i in range(torch.cuda.device_count()))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}] {p.name}  sm_{p.major}{p.minor}  {p.total_memory/2**30:.1f} GiB")
print(f"\naggregate GPU memory: {total/2**30:.1f} GiB")

torch          2.13.0+cu126
transformers   5.14.1
CUDA available True


GPUs visible   4
  GPU[0] NVIDIA GH200 120GB  sm_90  95.0 GiB
  GPU[1] NVIDIA GH200 120GB  sm_90  95.0 GiB
  GPU[2] NVIDIA GH200 120GB  sm_90  95.0 GiB
  GPU[3] NVIDIA GH200 120GB  sm_90  95.0 GiB

aggregate GPU memory: 380.0 GiB


## 2. Load the model

`nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` is a **hybrid** model: 88 layers
mixing Mamba2 state-space layers, a few attention layers, and a 512-expert MoE.
In BF16 the weights are about **225 GiB**, so:

| GPUs | Aggregate memory | Fits? |
|------|-----------------|-------|
| 1    | 95 GiB          | no    |
| 2    | 191 GiB         | no    |
| 4    | 382 GiB         | yes, with ~155 GiB spare |

`device_map="auto"` is what makes this work. It is **not** distributed training:
it is a single process, and `accelerate` simply places different layers on
different GPUs, passing activations between them with ordinary tensor copies.
There is no process group and no NCCL collective — which is exactly why this
notebook needs no NCCL module and no Slingshot configuration.

Expect the load to take **~2 minutes** (reading ~231 GB from the shared Lustre
filesystem). You will see a warning that the Mamba2 "fast path is not
available" — that is expected and fine; see the note after the next cell.


In [2]:
import time
from transformers import pipeline

MODEL = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"

t0 = time.time()
pipe = pipeline(
    "text-generation",
    model=MODEL,
    dtype=torch.bfloat16,   # note: `dtype`, not the older `torch_dtype`
    device_map="auto",      # shard the model across all visible GPUs
)
print(f"loaded in {time.time()-t0:.1f}s")

[transformers] The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/763 [00:00<?, ?it/s]

loaded in 111.5s


In [3]:
from collections import Counter

# Which GPU did each layer land on?
spread = Counter(pipe.model.hf_device_map.values())
print("layers per GPU:", dict(sorted(spread.items())))

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB used / {total/2**30:.1f} GiB")

layers per GPU: {0: 24, 1: 25, 2: 24, 3: 18}
  GPU[0]  62.8 GiB used / 95.0 GiB
  GPU[1]  67.1 GiB used / 95.0 GiB
  GPU[2]  67.1 GiB used / 95.0 GiB
  GPU[3]  52.1 GiB used / 95.0 GiB


> **About that "fast path is not available" warning.**
> The Mamba2 layers have optional hand-written CUDA kernels (`mamba-ssm`,
> `causal-conv1d`). Neither ships an ARM wheel, so installing them means
> compiling CUDA from source — slow and fragile, and deliberately out of scope
> here. `transformers` falls back to a pure-PyTorch implementation that is
> numerically correct, just slower. That is the right trade for a playground.
>
> If you later need the speed, that is when you reach for `module load cuda/12.6`
> and `export CC=/usr/bin/gcc-12` (the system `gcc` is 7.5 and too old).


## 3. Generating text

Two things about this model specifically.

**It is a reasoning model, and reasoning is ON by default.** Its chat template
opens a `<think>` block unless you say otherwise, so a short request spends its
whole token budget thinking out loud and you never see an answer. Passing
`enable_thinking=False` to `apply_chat_template` closes the block immediately.

**`enable_thinking` belongs to the chat template, not to `generate`.** Passing it
to the pipeline raises `ValueError: The following model_kwargs are not used by
the model`. So we render the prompt first, then generate from the string.


In [4]:
def chat(user_message, max_new_tokens=128, think=False, **kw):
    """Render the chat template, then generate. Returns the assistant's reply."""
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=think,      # a CHAT TEMPLATE kwarg, not a generate kwarg
    )
    out = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,             # reply only, not the prompt back
        clean_up_tokenization_spaces=False, # destructive for BPE tokenizers
        **kw,
    )
    return out[0]["generated_text"].strip()

> **Two warnings you will see on every call. Both are benign.**
>
> * `Both max_new_tokens (=128) and max_length (=20) seem to have been set.`
>   The model ships a `generation_config` carrying the library-default
>   `max_length=20`. Our `max_new_tokens` takes precedence, exactly as the
>   message says. Setting `generation_config.max_length = None` does *not*
>   silence it -- the config object coerces the value back to its default.
> * `Passing generation_config together with generation-related arguments is
>   deprecated.` A `transformers` housekeeping notice about the pipeline API,
>   not about anything you did.
>
> Learning to read past benign warnings -- and to recognise the ones that are
> *not* benign, like the `undefined symbol` failure in section 1 -- is most of
> what makes an HPC software stack tractable.


### Example 1 — a short factual explanation

In [5]:
t0 = time.time()
reply = chat("Explain what a supercomputer interconnect does, in two sentences.")
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A supercomputer interconnect is a high-speed network that links thousands of processors, memory units, and storage systems to enable rapid data exchange and coordinated computation across the entire system. It minimizes latency and maximizes bandwidth to ensure that parallel processing tasks can scale efficiently without bottlenecks.

[7.3s]


### Example 2 — something creative

In [6]:
t0 = time.time()
reply = chat("Write a haiku about GPUs waiting in a job queue.", max_new_tokens=64)
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Silent silicon,  
Queued in the humming dark — Waiting for their turn.

[2.5s]


### Example 3 — the same prompt, with reasoning switched on

This is the interesting one. Compare the output to what you would get with
`think=False`: the model narrates its own approach inside a `<think>` block
before answering. Useful when you want to see the reasoning; wasteful when you
just want the answer.


In [7]:
t0 = time.time()
reply = chat(
    "A job needs 12 GPUs. Each node has 4. How many nodes, and what if one node fails?",
    max_new_tokens=200,
    think=True,
)
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


We need to answer: job needs 12 GPUs, each node has 4 GPUs. Number of nodes needed = 12/4 = 3 nodes. If one node fails, then remaining nodes have 2 nodes * 4 = 8 GPUs, insufficient for 12 GPU job. So need additional node or job cannot run. Could say need 4 nodes total to tolerate one failure (redundancy). Or if one fails, you need to get another node to reach 12 GPUs. Provide answer.

</think>

**Baseline requirement**

- GPUs needed per job: **12**
- GPUs per node: **4**

\[
\text{Nodes required} = \frac{12\ \text{GPUs}}{4\ \text{GPUs/node}} = 3\ \text{nodes}
\]

So, under normal operation you would allocate **3 nodes** (each providing 4 GPUs) to satisfy

[27.1s]


## 4. Freeing the GPUs

The model holds ~225 GiB. Release it before loading anything else — or just
restart the kernel, which is more reliable.


In [8]:
import gc

# Dropping the reference is necessary but often not sufficient: Jupyter keeps
# its own references to cell outputs, so memory may not fall to zero here.
# Restarting the kernel is the reliable way to fully release the GPUs.
del pipe
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB still resident")

print("\nIf that is still large, restart the kernel to fully free the GPUs.")

  GPU[0]  57.6 GiB still resident
  GPU[1]  62.0 GiB still resident
  GPU[2]  62.0 GiB still resident
  GPU[3]  46.0 GiB still resident

If that is still large, restart the kernel to fully free the GPUs.


---

## Where to go next

- **Your job has a 24-hour walltime cap.** It is hard: the job dies at
  `24:00:00` and takes your tunnel with it. Save work to `/projects`, never to
  the node.
- **This notebook is single-node.** Models too large for one node (the 550B
  Nemotron Ultra, ~1.1 TB) need a multi-node serving engine such as vLLM with
  pipeline parallelism — a different exercise, and one where the Slingshot and
  NCCL configuration we skipped here suddenly matters a great deal.
- **Check the shared cache before downloading.** `/projects/a5k/public/hf/hub`
  likely already has the model you want, and the project storage quota is
  shared.
